<a href="https://colab.research.google.com/github/SergeiVKalinin/MSE_Fall_2026/blob/main/Module%205/Homework_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MSE 404/504 — Reinforcement Learning: Learning by Comparison

**Homework notebook | Run the experiments, compare algorithms, and explain what you observe.**

This assignment explores five elementary decision problems. In each, we keep the environment fixed and compare methods that differ in model knowledge, bootstrapping, exploration, or policy representation. **Your job is to execute the cells, inspect the plots and tables, and write short explanations in the answer boxes.**

**Instructions:** Run cells in order (Runtime → Run all in Colab). Keep `FAST=True` for a quick check; use `FAST=False` for a longer experiment where feasible. If a method does not converge, report that result rather than claiming success. Include your numerical outputs, note random variability, and answer each question in 3–6 sentences. No Gymnasium installation is required.

**Submission:** Share your completed `.ipynb` with outputs and your written answers. You may adjust training episodes, learning rates, exploration, or discretization, but record what you changed.

## Common methods — read once

**Dynamic programming (DP):** When the transition model is known, repeatedly apply the Bellman backup:

$$V_{k+1}(s)=\max_a\sum_{s'}P(s'|s,a)[R(s,a,s')+\gamma V_k(s')].$$

**Q-learning (off-policy):** Update the action value using the *best estimated* next action:

$$Q(s,a)\leftarrow Q(s,a)+\alpha[r+\gamma\max_b Q(s',b)-Q(s,a)].$$

**SARSA (on-policy):** Use the *actual next action* selected by the exploratory policy:

$$Q(s,a)\leftarrow Q(s,a)+\alpha[r+\gamma Q(s',a')-Q(s,a)].$$

**Monte Carlo (MC):** Estimate a fixed policy's value from complete episode returns $G_t$; **TD(0)** uses $r+\gamma V(s')$ to update after each transition. Compare both against the *same fixed policy*, not an optimal-control value.

**REINFORCE:** Optimize a stochastic policy directly from sampled returns using $\nabla_\theta\log\pi_\theta(a|s)G_t$. **Actor–critic** replaces complete-return feedback with a learned value estimate or TD advantage. For terminal transitions, the bootstrap term is zero.

**Control theory:** A physics-based controller uses known dynamics. For MountainCar, a bang-bang energy-pumping rule and numerical Bellman planning provide references; for CartPole, LQR solves a *linearized quadratic-cost* control problem, not the original nonlinear survival-reward objective.

**Interpretation:** A code cell completing successfully is not evidence that its learned policy has converged. Compare return, success, sample efficiency, and stability; distinguish a learned policy from a known-model reference.

## Setup and shared functions
The following code defines reproducible training, evaluation, and plotting helpers. Run it before the exercises.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd
from scipy.linalg import solve_discrete_are
import torch
from torch import nn
FAST=True
np.random.seed(5);torch.manual_seed(5)

In [ ]:
def binstate(s,lo,hi,bins):
 z=(np.clip(np.asarray(s),lo,hi)-lo)/(np.asarray(hi)-lo)
 return tuple(np.minimum((z*np.asarray(bins)).astype(int),np.asarray(bins)-1))
def train(make,lo,hi,bins,kind='q',episodes=300,seed=1):
 rng=np.random.default_rng(seed);e=make();Q=np.zeros(tuple(bins)+(e.na,),np.float32);returns=[];success=[];steps=0
 def idx(s):return binstate(s,lo,hi,bins)
 def action(s,eps):return int(rng.integers(e.na)) if rng.random()<eps else int(Q[idx(s)].argmax())
 for ep in range(episodes):
  s=e.reset(int(rng.integers(1e9)));eps=.03+.97*np.exp(-5*ep/episodes);a=action(s,eps);R=0
  for t in range(e.limit):
   sn,r,done,trunc=e.step(a);an=action(sn,eps);steps+=1;R+=r
   target=r if done or trunc else r+.99*(Q[idx(sn)].max() if kind=='q' else Q[idx(sn)][an])
   Q[idx(s)][a]+=.2*(target-Q[idx(s)][a]);s,a=sn,an
   if done or trunc:break
  returns.append(R);success.append(e.success)
 return {'policy':lambda s:int(Q[idx(s)].argmax()),'returns':np.array(returns),'success':np.array(success),'Q':Q,'steps':steps}
def test(make,policies,n=8):
 rows=[];paths={}
 for name,policy in policies.items():
  rr=[];win=[];length=[]
  for seed in range(n):
   e=make();s=e.reset(seed+1000);path=[np.array(s)];R=0
   for t in range(e.limit):
    s,r,d,tr=e.step(int(policy(s)));R+=r;path.append(np.array(s))
    if d or tr:break
   rr.append(R);win.append(e.success);length.append(t+1)
   if seed==0:paths[name]=np.array(path)
  rows.append([name,np.mean(rr),np.std(rr),np.mean(win),np.mean(length)])
 return pd.DataFrame(rows,columns=['Method','Mean reward','Reward SD','Success rate','Mean steps']),paths
def plotpaths(paths,i,j,title):
 plt.figure(figsize=(7,3.5))
 for name,path in paths.items():plt.plot(path[:,i],path[:,j],label=name)
 plt.xlabel(f'state[{i}]');plt.ylabel(f'state[{j}]');plt.title(title);plt.legend();plt.show()
def learning(models,title):
 plt.figure(figsize=(7,3))
 for name,v in models.items():
  r=v['returns'];w=min(30,len(r));plt.plot(np.convolve(r,np.ones(w)/w,'valid'),label=name)
 plt.title(title);plt.xlabel('Episodes');plt.ylabel('Rolling return');plt.legend();plt.show()
def reinforce(make,scale,episodes=80,seed=3):
 rng=np.random.default_rng(seed);e=make();torch.manual_seed(seed);scale=np.asarray(scale)
 net=nn.Sequential(nn.Linear(len(scale),24),nn.Tanh(),nn.Linear(24,24),nn.Tanh(),nn.Linear(24,e.na));opt=torch.optim.Adam(net.parameters(),lr=.002)
 returns=[];success=[];steps=0;baseline=0
 for ep in range(episodes):
  s=e.reset(int(rng.integers(1e9)));logs=[];rews=[]
  for t in range(e.limit):
   dist=torch.distributions.Categorical(logits=net(torch.tensor(s/scale,dtype=torch.float32)))
   a=dist.sample();logs.append(dist.log_prob(a));s,r,d,tr=e.step(int(a));rews.append(r);steps+=1
   if d or tr:break
  G=0;g=[]
  for r in rews[::-1]:G=r+.99*G;g.append(G)
  g=np.array(g[::-1]);baseline=.95*baseline+.05*g.mean()
  adv=torch.tensor((g-baseline)/(g.std()+1e-5),dtype=torch.float32)
  loss=-(torch.stack(logs)*adv).mean();opt.zero_grad();loss.backward();opt.step()
  returns.append(sum(rews));success.append(e.success)
 def policy(s):
  with torch.no_grad():return int(net(torch.tensor(np.asarray(s)/scale,dtype=torch.float32)).argmax())
 return {'policy':policy,'returns':np.array(returns),'success':np.array(success),'steps':steps}


# Exercise 1 — 3×4 Gridworld: Known model vs. learned model-free control

**Purpose:** Determine whether Q-learning can recover a policy similar to value iteration when it sees only sampled transitions.

The grid has a blocked cell, terminal rewards +1 and −1, a small step penalty, and stochastic actions (intended move 80%, sideways moves 10% each).

**Run:** Calculate the DP reference and train Q-learning. Examine the two value maps and the evaluation results.

In [ ]:
class Grid34:
 na=4;limit=80;wall=(1,1);term={(0,3):1.,(1,3):-1.};moves=[(-1,0),(0,1),(1,0),(0,-1)]
 def reset(self,seed=0):self.rng=np.random.default_rng(seed);self.s=(2,0);self.t=0;self.success=False;return np.array(self.s)
 def next(self,s,a):
  z=(s[0]+self.moves[a][0],s[1]+self.moves[a][1]);return s if z[0]<0 or z[0]>=3 or z[1]<0 or z[1]>=4 or z==self.wall else z
 def step(self,a):
  a=int(self.rng.choice([(a-1)%4,a,(a+1)%4],p=[.1,.8,.1]));self.s=self.next(self.s,a);self.t+=1
  self.success=self.s==(0,3);r=self.term.get(self.s,-.04)
  return np.array(self.s),r,self.s in self.term,self.t>=self.limit

def grid_dp(make,shape,terminals,discount=.99):
 e=make();V=np.zeros(shape);P=np.zeros(shape,int)
 for it in range(1000):
  Q=np.full(shape+(e.na,),-1e6,dtype=float)
  for s in np.ndindex(shape):
   if s in terminals or (hasattr(e,'wall') and s==e.wall):Q[s]=0;continue
   for a in range(e.na):
    options=[((a-1)%4,.1),(a,.8),((a+1)%4,.1)]
    Q[s+(a,)]=sum(prob*(e.term.get(ns,-.04)+discount*(0 if ns in terminals else V[ns])) for b,prob in options for ns in [e.next(s,b)])
  W=Q.max(-1)
  if np.max(abs(W-V))<1e-9:break
  V=W;P=Q.argmax(-1)
 return V,P
v34,p34=grid_dp(Grid34,(3,4),Grid34.term)
print('DP values:');print(np.round(v34,2));print('Policy (0=up,1=right,2=down,3=left):');print(p34)
gq=train(Grid34,[0,0],[2,3],(3,4),'q',300 if FAST else 3000)
gs=train(Grid34,[0,0],[2,3],(3,4),'sarsa',300 if FAST else 3000,seed=2)
display(test(Grid34,{'DP':lambda s:p34[tuple(s)],'Q':gq['policy'],'SARSA':gs['policy']})[0])

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(9,3))
ax[0].imshow(v34);ax[0].set_title('Known-model DP value')
ax[1].imshow(gq['Q'].max(-1));ax[1].set_title('Q-learning value estimate')
plt.tight_layout();plt.show()
learning({'Q-learning':gq,'SARSA extension':gs},'3×4 learning curves')

**Your interpretation**

1. How do DP and Q-learning differ in what information they require?
2. How closely do the learned values and policies agree? Cite a numerical result or a visible difference.
3. Would increasing the training budget necessarily improve the agreement? Explain.

**Answer:**

_Write your answer here._

# Exercise 2 — FrozenLake: Monte Carlo vs. TD(0)

**Purpose:** Compare full-return estimation and one-step bootstrapping under stochastic transitions.

The standard 4×4 map contains holes and a goal. **The lake is slippery:** each selected direction results in the intended or either perpendicular direction with probability $1/3$. The code implements this directly, without relying on Gymnasium's `is_slippery` API.

**Run:** Compute the optimal DP map, then evaluate one *fixed exploratory policy* using MC and TD(0). Examine the value-error plot against the exact value of that fixed policy.

In [ ]:
class FrozenLake:
 na=4;limit=100;layout=['SFFF','FHFH','FFFH','HFFG'];moves=[(0,-1),(1,0),(0,1),(-1,0)]
 def __init__(self,slippery=True):self.slippery=slippery
 def reset(self,seed=0):self.rng=np.random.default_rng(seed);self.s=(0,0);self.t=0;self.success=False;return np.array(self.s)
 def next(self,s,a):
  dr,dc=self.moves[a];return (min(3,max(0,s[0]+dr)),min(3,max(0,s[1]+dc)))
 def step(self,a):
  if self.slippery:a=int(self.rng.choice([(a-1)%4,a,(a+1)%4]))
  self.s=self.next(self.s,a);self.t+=1;cell=self.layout[self.s[0]][self.s[1]]
  self.success=cell=='G';return np.array(self.s),float(self.success),cell in 'HG',self.t>=self.limit

def lake_dp(slippery=True):
 e=FrozenLake(slippery);V=np.zeros((4,4));P=np.zeros((4,4),int)
 for it in range(2000):
  Q=np.zeros((4,4,4))
  for s in np.ndindex(4,4):
   if e.layout[s[0]][s[1]] in 'HG':continue
   for a in range(4):
    choices=[((a-1)%4,1/3),(a,1/3),((a+1)%4,1/3)] if slippery else [(a,1)]
    for b,pr in choices:
     ns=e.next(s,b);cell=e.layout[ns[0]][ns[1]]
     Q[s+(a,)]+=pr*(float(cell=='G')+.99*(0 if cell in 'HG' else V[ns]))
  W=Q.max(-1)
  if np.max(abs(W-V))<1e-10:break
  V=W;P=Q.argmax(-1)
 return V,P
vl,pl=lake_dp(True);vd,p_det=lake_dp(False)
print('Slippery values:');print(np.round(vl,3));print('Deterministic values:');print(np.round(vd,3))
print('Example: intended RIGHT from (0,0) yields UP, RIGHT, DOWN each with probability 1/3.')
lq=train(lambda:FrozenLake(True),[0,0],[3,3],(4,4),'q',1200 if FAST else 9000)
ls=train(lambda:FrozenLake(True),[0,0],[3,3],(4,4),'sarsa',1200 if FAST else 9000,seed=2)
learning({'Q':lq,'SARSA':ls},'Slippery FrozenLake learning')
display(test(lambda:FrozenLake(True),{'DP':lambda s:pl[tuple(s)],'Q':lq['policy'],'SARSA':ls['policy']},n=80)[0])

In [ ]:
# Exact fixed-policy evaluation and first-visit MC versus TD(0)
pi=np.full((4,4,4),0.35/4)
for state in np.ndindex(4,4):pi[state+(int(pl[state]),)]+=0.65
def eval_pi(pi,gamma=.99):
 e=FrozenLake(True);v=np.zeros((4,4))
 for it in range(5000):
  w=v.copy()
  for s in np.ndindex(4,4):
   if e.layout[s[0]][s[1]] in 'HG':w[s]=0;continue
   w[s]=0
   for a in range(4):
    for b in ((a-1)%4,a,(a+1)%4):
     sn=e.next(s,b);cell=e.layout[sn[0]][sn[1]]
     w[s]+=pi[s+(a,)]/3*(float(cell=='G')+(0 if cell in 'HG' else gamma*v[sn]))
  if np.max(abs(w-v))<1e-11:break
  v=w
 return w
reference=eval_pi(pi)
def mc_td(episodes=600,alpha=.07,seed=8):
 rng=np.random.default_rng(seed);mc=np.zeros((4,4));td=np.zeros((4,4));visits=np.zeros((4,4));errors=[]
 for ep in range(episodes):
  env=FrozenLake(True);s=env.reset(int(rng.integers(1000000000)));traj=[]
  for t in range(env.limit):
   a=int(rng.choice(4,p=pi[tuple(s)]));sn,r,done,tr=env.step(a)
   old=tuple(s);new=tuple(sn);td[old]+=alpha*(r+(0 if done or tr else .99*td[new])-td[old]);traj.append((old,r));s=sn
   if done or tr:break
  g=0;returns=[]
  for state,r in traj[::-1]:g=r+.99*g;returns.append((state,g))
  seen=set()
  for state,g in returns[::-1]:
   if state in seen:continue
   seen.add(state);visits[state]+=1;mc[state]+=(g-mc[state])/visits[state]
  if (ep+1)%20==0:
   mask=np.array([[z not in 'HG' for z in row] for row in env.layout])
   errors.append([ep+1,np.sqrt(np.mean((mc[mask]-reference[mask])**2)),np.sqrt(np.mean((td[mask]-reference[mask])**2))])
 return mc,td,np.array(errors)
mc,td,errors=mc_td(500 if FAST else 5000)
fig,ax=plt.subplots(1,3,figsize=(11,3))
for a,z,name in zip(ax,[reference,mc,td],['Exact fixed-policy DP','Monte Carlo','TD(0)']):
 a.imshow(z,vmin=0,vmax=max(.1,reference.max()));a.set_title(name)
plt.tight_layout();plt.show()
plt.figure(figsize=(7,3));plt.plot(errors[:,0],errors[:,1],label='MC');plt.plot(errors[:,0],errors[:,2],label='TD(0)')
plt.xlabel('Episodes');plt.ylabel('Value RMSE');plt.legend();plt.show()
print('Final RMSE, MC / TD:',np.round(errors[-1,1:],4))

**Your interpretation**

1. Why is FrozenLake different from a deterministic shortest-path problem?
2. What does MC need to observe before updating? What does TD(0) need?
3. Which method has lower error in your run? Does one run prove it is always better?

**Answer:**

_Write your answer here._

# Exercise 3 — CliffWalking: SARSA vs. Q-learning

**Purpose:** Test how on-policy and off-policy updates respond to risky exploration.

Stepping off the cliff produces a large penalty and returns the agent to the start. Train both methods with the same environment and inspect their rewards, routes, and cliff-fall counts.

In [ ]:
class Cliff:
 na=4;limit=250;moves=[(-1,0),(0,1),(1,0),(0,-1)]
 def reset(self,seed=0):self.s=(3,0);self.t=0;self.success=False;self.falls=0;return np.array(self.s)
 def step(self,a):
  dr,dc=self.moves[a];r,c=self.s;ns=(min(3,max(0,r+dr)),min(11,max(0,c+dc)));self.t+=1
  reward=-1.
  if ns[0]==3 and 1<=ns[1]<=10:reward=-100.;ns=(3,0);self.falls+=1
  self.s=ns;self.success=ns==(3,11)
  return np.array(ns),reward,self.success,self.t>=self.limit
clq=train(Cliff,[0,0],[3,11],(4,12),'q',350 if FAST else 2500)
cls=train(Cliff,[0,0],[3,11],(4,12),'sarsa',350 if FAST else 2500,seed=2)
learning({'Q':clq,'SARSA':cls},'CliffWalking training returns')
display(test(Cliff,{'Q':clq['policy'],'SARSA':cls['policy']})[0])
print('Q policy map:');print(clq['Q'].argmax(-1));print('SARSA policy map:');print(cls['Q'].argmax(-1))

In [ ]:
for name,model in [('Q-learning',clq),('SARSA',cls)]:
 falls=[];returns=[]
 for seed in range(15):
  e=Cliff();s=e.reset(seed);R=0
  for t in range(e.limit):
   s,r,d,tr=e.step(model['policy'](s));R+=r
   if d or tr:break
  falls.append(e.falls);returns.append(R)
 print(name,'greedy mean return',round(np.mean(returns),2),'mean falls',round(np.mean(falls),2))

**Your interpretation**

1. Which method performs better during exploratory training, and which under greedy evaluation?
2. How does the difference between $Q(s',a')$ and $\max_a Q(s',a)$ explain the behavior?
3. What changes would you expect if exploration probability were increased?

**Answer:**

_Write your answer here._

# Exercise 4 — MountainCar: Physics-based control vs. model-free RL

**Purpose:** See why the fastest route to a goal can begin by moving away from it.

The state is $(x,v)$ and actions accelerate left, coast, or accelerate right. The physics-based references use a velocity-sign bang-bang rule and numerical Bellman planning; Q-learning, SARSA, and REINFORCE learn from experience. The reference DP is numerical, not an analytical closed-form solution.

$$v_{t+1}=\operatorname{clip}[v_t+0.001(a_t-1)-0.0025\cos(3x_t),-0.07,0.07].$$

**Run:** Examine the calculated value map and policy, training curves, success rates, and $(x,v)$ trajectories. If the short training budget is insufficient, increase it and document the change.

In [ ]:
class MountainCar:
 na=3;limit=200
 @staticmethod
 def dyn(s,a):
  x,v=s;v=np.clip(v+.001*(a-1)-.0025*np.cos(3*x),-.07,.07);x=np.clip(x+v,-1.2,.6)
  if x<=-1.2 and v<0:v=0
  return np.array([x,v])
 def reset(self,seed=0):
  self.s=np.array([np.random.default_rng(seed).uniform(-.6,-.4),0.]);self.t=0;self.success=False;return self.s.copy()
 def step(self,a):
  self.s=self.dyn(self.s,a);self.t+=1;self.success=self.s[0]>=.5
  return self.s.copy(),-1.,self.success,self.t>=self.limit
from scipy.ndimage import map_coordinates
X=np.linspace(-1.2,.6,101);Y=np.linspace(-.07,.07,71);xx,vv=np.meshgrid(X,Y,indexing='ij')
trans=[]
for a in range(3):
 vn=np.clip(vv+.001*(a-1)-.0025*np.cos(3*xx),-.07,.07);xn=np.clip(xx+vn,-1.2,.6);vn=np.where((xn<=-1.2)&(vn<0),0,vn);trans.append((xn,vn))
def interp(V,x,v):return map_coordinates(V,[(x+1.2)/1.8*(len(X)-1),(v+.07)/.14*(len(Y)-1)],order=1,mode='nearest')
V=np.zeros_like(xx)
for k in range(350):
 Q=np.stack([np.where(xx>=.5,0,-1+.99*interp(V,*z)) for z in trans],axis=-1);Vnew=Q.max(-1)
 if np.max(abs(Vnew-V))<1e-5:break
 V=Vnew
Q=np.stack([np.where(xx>=.5,0,-1+.99*interp(V,*z)) for z in trans],axis=-1);P=Q.argmax(-1)
def dp(s):return int(P[np.clip(np.searchsorted(X,s[0]),0,len(X)-1),np.clip(np.searchsorted(Y,s[1]),0,len(Y)-1)])
def bang(s):return 2 if s[1]>=0 else 0
def mpc(s):
 # Short-horizon receding optimization with DP terminal value
 candidates=[(0,np.asarray(s),2)]
 for h in range(4):
  nxt=[]
  for score,z,first in candidates:
   for a in range(3):
    zn=MountainCar.dyn(z,a);i=np.clip(np.searchsorted(X,zn[0]),0,len(X)-1);j=np.clip(np.searchsorted(Y,zn[1]),0,len(Y)-1)
    nxt.append((-(h+1)+(.99**(h+1))*V[i,j],zn,a if h==0 else first))
  nxt.sort(key=lambda z:z[0],reverse=True);candidates=nxt[:7]
 return int(candidates[0][2])
fig,ax=plt.subplots(1,2,figsize=(10,3));ax[0].imshow(V.T,origin='lower',extent=[-1.2,.6,-.07,.07],aspect='auto');ax[0].set_title('DP value V(x,v)');ax[1].imshow(P.T,origin='lower',extent=[-1.2,.6,-.07,.07],aspect='auto');ax[1].set_title('DP policy');plt.show()
display(test(MountainCar,{'Bang-bang':bang,'Bellman':dp,'MPC':mpc},4)[0])

In [ ]:
mq=train(MountainCar,[-1.2,-.07],[.6,.07],(28,20),'q',120 if FAST else 1200)
ms=train(MountainCar,[-1.2,-.07],[.6,.07],(28,20),'sarsa',120 if FAST else 1200,seed=2)
mr=reinforce(MountainCar,[1.2,.07],35 if FAST else 600)
learning({'Q':mq,'SARSA':ms,'REINFORCE':mr},'MountainCar learning')
report,paths=test(MountainCar,{'DP':dp,'Q':mq['policy'],'SARSA':ms['policy'],'REINFORCE':mr['policy']});display(report);plotpaths(paths,0,1,'MountainCar phase space')

**Your interpretation**

1. Why does accelerating in the direction of motion help the car build energy?
2. Compare the bang-bang, DP, and learned controllers. Which reach the goal reliably? Cite the observed results.
3. What do the phase-space trajectories reveal that a position-versus-time plot would hide?
4. If an RL method fails, identify one likely reason (exploration, state resolution, sparse reward, or training budget).

**Answer:**

_Write your answer here._

# Exercise 5 — CartPole: LQR vs. policy learning

**Purpose:** Compare a model-based stabilizing controller with learned policies in a continuous four-dimensional state space.

The state is $(x,\dot x,\theta,\dot\theta)$. LQR uses the linearization near the upright equilibrium and a quadratic cost. REINFORCE and actor–critic learn stochastic policies from trajectories. The RL environment instead rewards survival, so these are **not identical optimization objectives**.

**Run:** Inspect the LQR controller, training curves, evaluation table, and angular phase-space trajectories. Then run the additional actor–critic example.

In [ ]:
class CartPole:
 na=2;limit=500
 @staticmethod
 def rhs(s,F):
  x,xd,th,thd=s;c=np.cos(th);z=np.sin(th);temp=(F+.05*thd**2*z)/1.1
  thacc=(9.8*z-c*temp)/(.5*(4/3-.1*c*c/1.1));xacc=temp-.05*thacc*c/1.1
  return np.array([xd,xacc,thd,thacc])
 @staticmethod
 def dyn(s,a):return np.asarray(s)+.02*CartPole.rhs(s,10 if a else -10)
 def reset(self,seed=0):
  self.s=np.random.default_rng(seed).uniform(-.05,.05,4);self.t=0;self.success=False;return self.s.copy()
 def step(self,a):
  self.s=self.dyn(self.s,a);self.t+=1
  fail=abs(self.s[0])>2.4 or abs(self.s[2])>.20944;self.success=not fail and self.t>=self.limit
  return self.s.copy(),1.,fail,self.t>=self.limit
h=1e-5;eye=np.eye(4)
A=np.column_stack([(CartPole.rhs(h*eye[i],0)-CartPole.rhs(-h*eye[i],0))/(2*h) for i in range(4)])
B=((CartPole.rhs(np.zeros(4),h)-CartPole.rhs(np.zeros(4),-h))/(2*h)).reshape(4,1)
Ad=np.eye(4)+.02*A;Bd=.02*B;R=np.array([[.1]]);P_lqr=solve_discrete_are(Ad,Bd,np.diag([1,1,12,2]),R)
K=np.linalg.solve(Bd.T@P_lqr@Bd+R,Bd.T@P_lqr@Ad)
def lqr(s):return int(float((-K@np.asarray(s))[0])>=0)
print('LQR gain:',K);report,paths=test(CartPole,{'LQR':lqr});display(report);plotpaths(paths,2,3,'CartPole angular phase space')

In [ ]:
cq=train(CartPole,[-2.4,-3,-.21,-3],[2.4,3,.21,3],(6,6,10,10),'q',150 if FAST else 1300)
cs=train(CartPole,[-2.4,-3,-.21,-3],[2.4,3,.21,3],(6,6,10,10),'sarsa',150 if FAST else 1300,seed=2)
cr=reinforce(CartPole,[2.4,3,.21,3],70 if FAST else 700)
learning({'Q':cq,'SARSA':cs,'REINFORCE':cr},'CartPole learning')
report,paths=test(CartPole,{'LQR':lqr,'Q':cq['policy'],'SARSA':cs['policy'],'REINFORCE':cr['policy']});display(report);plotpaths(paths,2,3,'CartPole policy comparison')

In [ ]:
# One-step actor–critic: explicit, from scratch (small demonstration budget)
def actor_critic_cartpole(episodes=70,seed=19):
 torch.manual_seed(seed);rng=np.random.default_rng(seed)
 actor=nn.Sequential(nn.Linear(4,32),nn.Tanh(),nn.Linear(32,2))
 critic=nn.Sequential(nn.Linear(4,32),nn.Tanh(),nn.Linear(32,1))
 oa=torch.optim.Adam(actor.parameters(),lr=.002)
 oc=torch.optim.Adam(critic.parameters(),lr=.004)
 scale=np.array([2.4,3,.21,3.])
 hist=[]
 for ep in range(episodes):
  env=CartPole();s=env.reset(int(rng.integers(1e9)));total=0
  for t in range(env.limit):
   st=torch.tensor(s/scale,dtype=torch.float32)
   dist=torch.distributions.Categorical(logits=actor(st))
   a=dist.sample()
   sn,r,done,tr=env.step(int(a.item()));total+=r
   nxt=torch.tensor(sn/scale,dtype=torch.float32)
   target=torch.tensor(r,dtype=torch.float32)
   with torch.no_grad():
    if not(done or tr):target=target+.99*critic(nxt).squeeze()
   value=critic(st).squeeze();adv=target-value
   loss_c=.5*adv.square()
   oc.zero_grad();loss_c.backward();oc.step()
   loss_a=-dist.log_prob(a)*adv.detach()
   oa.zero_grad();loss_a.backward();oa.step()
   s=sn
   if done or tr:break
  hist.append(total)
 def policy(s):
  with torch.no_grad():return int(actor(torch.tensor(np.asarray(s)/scale,dtype=torch.float32)).argmax())
 return {'policy':policy,'returns':np.array(hist),'steps':int(sum(hist))}
ac=actor_critic_cartpole(25 if FAST else 300)
learning({'REINFORCE':cr,'Actor–critic':ac},'CartPole policy-gradient comparison')
display(test(CartPole,{'LQR':lqr,'REINFORCE':cr['policy'],'Actor–critic':ac['policy']},n=12)[0])

**Your interpretation**

1. Why does LQR work best near the upright equilibrium, and why is its optimality only local for the nonlinear system?
2. Compare REINFORCE and actor–critic. What learning signal does each use?
3. Which controllers stabilized the pole in your runs? Use measured survival times, not assumptions.
4. Why does tabular state discretization become difficult in CartPole?

**Answer:**

_Write your answer here._

# Final comparison — Which algorithm would you choose?

Complete a brief summary using your actual outputs. You do **not** need to reproduce every plot.

| Environment | Comparison | Better-performing method in your run | Evidence (return, error, success, or steps) |
|---|---|---|---|
| 3×4 Gridworld | DP vs. Q-learning | | |
| FrozenLake | MC vs. TD(0) | | |
| CliffWalking | SARSA vs. Q-learning | | |
| MountainCar | Physics-based / DP vs. learned RL | | |
| CartPole | LQR vs. policy-gradient RL | | |

**Concluding question:** Suppose you are controlling an expensive scientific instrument. When would you use known-model optimal control rather than model-free RL? Discuss model accuracy, cost of experiments, exploration risk, and the need to adapt.

**Answer:**

_Write your answer here._